 ### GaurdRails
 
  GaurdRails are saftey mechanisim that control what goes into and comes outof a AI agent.
    Its ensure that
					1. Only process safe, appropriate action
					2. Only perform approved actions
					3. Only return validated, compliant output

### HumanInTheLoopMiddleware

Human-in-the-loop (HITL) middleware pauses the agent right before a sensitive tool runs, saves state in a checkpointer, and waits for a human decision. After you resume with Command(resume=...), the agent continues—or skips the tool if rejected.

In [ ]:
"""
HumanInTheLoopMiddleware demo with Groq + dummy tools + manual tests.

Run: python hitl_groq_demo.py
"""

import os
import sys
import uuid
from dotenv import load_dotenv

load_dotenv()

if not os.getenv("GROQ_API_KEY"):
    sys.exit("Set GROQ_API_KEY in .env")

from langchain.agents import create_agent
from langchain.agents.middleware import HumanInTheLoopMiddleware
from langchain.tools import tool
from langchain_groq import ChatGroq
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command

# ---------------------------------------------------------------------------
# Dummy tools — names must match keys in interrupt_on
# ---------------------------------------------------------------------------

@tool
def get_weather(city: str) -> str:
    """Get weather for a city. Safe — no approval needed."""
    return f"Weather in {city}: sunny, 22°C."

@tool
def transfer_money(amount: float, to_account: str) -> str:
    """Transfer money to another account. Sensitive — requires approval."""
    return f"Transferred ${amount:.2f} to account {to_account}."

@tool
def delete_user_data(user_id: str) -> str:
    """Permanently delete all data for a user. Destructive — requires approval."""
    return f"Deleted all data for user_id={user_id}."

TOOLS = [get_weather, transfer_money, delete_user_data]

SYSTEM_PROMPT = (
    "You are a helpful assistant. Use tools when the user asks. "
    "Be concise."
)

def build_agent():
    model = ChatGroq(
        model="openai/gpt-oss-20b",
        temperature=0,
        api_key=os.getenv("GROQ_API_KEY"),
    )

    return create_agent(
        model=model,
        tools=TOOLS,
        system_prompt=SYSTEM_PROMPT,
        checkpointer=InMemorySaver(),  # required for HITL
        middleware=[
            HumanInTheLoopMiddleware(
                interrupt_on={
                    "get_weather": False,  # auto-approved
                    "transfer_money": True,  # approve | edit | reject | respond
                    "delete_user_data": {
                        "allowed_decisions": ["approve", "reject"],
                    },
                },
                description_prefix="Approval required before running tool",
            ),
        ],
    )

def new_thread_config():
    return {"configurable": {"thread_id": str(uuid.uuid4())}}

def has_interrupt(result) -> bool:
    if getattr(result, "interrupts", None):
        return len(result.interrupts) > 0
    if isinstance(result, dict) and result.get("__interrupt__"):
        return True
    return False

def print_interrupt_details(result):
    if getattr(result, "interrupts", None):
        intr = result.interrupts[0]
        value = intr.value if hasattr(intr, "value") else intr
        print("INTERRUPT:", value)
        return
    if isinstance(result, dict):
        print("INTERRUPT:", result.get("__interrupt__"))

def invoke_user(agent, text: str, config):
    return agent.invoke(
        {"messages": [{"role": "user", "content": text}]},
        config=config,
        version="v2",
    )

def resume_decisions(agent, decisions, config):
    return agent.invoke(
        Command(resume={"decisions": decisions}),
        config=config,
        version="v2",
    )

def last_ai_text(result) -> str:
    messages = getattr(result, "messages", None) or result.get("messages", [])
    for msg in reversed(messages):
        role = getattr(msg, "type", None) or (msg.get("role") if isinstance(msg, dict) else None)
        if role in ("ai", "assistant"):
            content = getattr(msg, "content", None) or msg.get("content", "")
            return content if isinstance(content, str) else str(content)
    return str(result)

# ---------------------------------------------------------------------------
# Tests
# ---------------------------------------------------------------------------

def test_safe_tool_no_interrupt():
    """get_weather: interrupt_on=False → should complete without HITL pause."""
    print("\n" + "=" * 70)
    print("TEST 1: Safe tool (no interrupt)")
    agent = build_agent()
    config = new_thread_config()
    result = invoke_user(agent, "What is the weather in Paris?", config)
    assert not has_interrupt(result), "Expected no interrupt for get_weather"
    print("PASS — no interrupt")
    print("Reply:", last_ai_text(result)[:200])

def test_transfer_approve():
    """transfer_money: interrupt → human approves → tool runs."""
    print("\n" + "=" * 70)
    print("TEST 2: transfer_money — APPROVE")
    agent = build_agent()
    config = new_thread_config()
    result = invoke_user(
        agent,
        "Transfer 250 dollars to account ACC-999 using transfer_money.",
        config,
    )
    assert has_interrupt(result), "Expected interrupt before transfer_money"
    print_interrupt_details(result)
    result2 = resume_decisions(agent, [{"type": "approve"}], config)
    assert not has_interrupt(result2), "Expected run to finish after approve"
    print("PASS — approved and continued")
    print("Reply:", last_ai_text(result2)[:300])

def test_transfer_reject():
    """transfer_money: interrupt → human rejects → tool should not run as requested."""
    print("\n" + "=" * 70)
    print("TEST 3: transfer_money — REJECT")
    agent = build_agent()
    config = new_thread_config()
    result = invoke_user(
        agent,
        "Please transfer 1000 to account HACK-001.",
        config,
    )
    assert has_interrupt(result), "Expected interrupt"
    result2 = resume_decisions(
        agent,
        [
            {
                "type": "reject",
                "message": "Unauthorized transfer. Do not retry.",
            }
        ],
        config,
    )
    print("PASS — rejected")
    print("Reply:", last_ai_text(result2)[:300])

def test_delete_edit_then_approve():
    """delete_user_data: edit args (lower user_id) then approve — if edit supported."""
    print("\n" + "=" * 70)
    print("TEST 4: delete_user_data — EDIT + APPROVE (optional)")
    agent = build_agent()
    config = new_thread_config()
    result = invoke_user(
        agent,
        "Delete all data for user_id PROD-USER-42.",
        config,
    )
    if not has_interrupt(result):
        print("SKIP — model did not call delete_user_data (LLM non-determinism)")
        return
    print_interrupt_details(result)
    # Edit: only allowed if interrupt_on includes "edit" — delete_user_data only has approve/reject
    # So we only approve here
    result2 = resume_decisions(agent, [{"type": "approve"}], config)
    print("Reply:", last_ai_text(result2)[:300])

def main():
    test_safe_tool_no_interrupt()
    test_transfer_approve()
    test_transfer_reject()
    test_delete_edit_then_approve()
    print("\n" + "=" * 70)
    print("All HITL tests finished.")

if __name__ == "__main__":
    main()